This notebook walks through how to make graphs in this repo. if you don't know how to set up a git repo locally, try reading thru the README.md first. You can copy this example.ipynb and rename it for your own graphing work. <b> Please do each graph in a seperate notebook!</b>

1. Import the appropriate packages. We'll use plotly.graph_objects for making all of our graphs.

In [54]:
import datetime as dt
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

2. We'll get data using the fredapi package. Set the path to the text file with your API key

In [55]:
from fredapi import Fred

API_KEY_PATH = "fred_api_key.txt" 
fred = Fred(api_key_file = API_KEY_PATH) 

3. Set the fed_2025 template as default

In [56]:
import graph_templates

pio.templates.default = 'fed_2025'

# Call the graph the exact same thing as its notebook (minus the ipynb suffix)
GRAPH_NAME = "unemployment_quits"

# Now is a good time to set the path to the graph output folder!
GRAPH_OUTPUT_PATH = "../graph_output"


4. Use the fredapi to get the data and prepare it for graphing. Documentation on the optional parameters that can be passed to the get_series called are found here (the documentation in fredapi is out of date). 

https://fred.stlouisfed.org/docs/api/fred/series_observations.html#Description

If you get data from somewhere else thats fine too! Put the raw csv in the "raw_data" folder and read it in here. Make sure not to edit the raw data, just transform and graph it.

In [57]:
# The get_series_info method may be useful
fred.get_series_info(series_id="CPIAUCSL")

# It's good practice to store the series codes in a dictionary with their names
series_codes = {
    "U6 Unemployment": "U6RATE",
    "U2 Unemployment": "U2RATE",
    "Fixed Rate 30 Year Mortgage Average": "MORTGAGE30US",
}

In [58]:
today = dt.date.today()

u6_unemp = fred.get_series(
    series_id=series_codes["U6 Unemployment"],
    observation_start=dt.date(2020, 1, 1),
    observation_end=today,
    units='pc1'
).rename("U6 Unemployment")

u2_unemp = fred.get_series(
    series_id=series_codes["U2 Unemployment"],
    observation_start=dt.date(2020, 1, 1),
    observation_end=today,
    units='pc1',
).rename("U2 Unemployment")

morg_deli = fred.get_series(
    series_id=series_codes["Fixed Rate 30 Year Mortgage Average"],
    observation_start=dt.date(2020, 1, 1),
    observation_end=today,
    units='pc1'
).rename("Fixed Rate 30 Year Mortgage")

joined_df = pd.concat(
    [u6_unemp, u2_unemp, morg_deli], # srs to join
    axis=1, # multiple observations per index entry
    join='inner' # only include observations where none of the 3 are missing
)

# Naming the index makes it easier tot title
joined_df.index.name = "Date"

joined_df.tail()

,U6 Unemployment,U2 Unemployment,Fixed Rate 30 Year Mortgage
Date,,,
2022-12-01,-9.58904,-15.78947,108.68167
2023-06-01,2.98507,6.25000,33.39882
2024-02-01,7.35294,11.76471,8.86700
2024-08-01,9.85915,11.11111,-2.46377
2025-05-01,5.40541,5.26316,-6.37119


5. Now that all our data is ready, make the graph and have it save itself as a .html file to graph_output whenver the notebooks is rerun. The name of the file should exactly match the notebook name. For instance, this file "example.ipynb" produces the graph "example.html." Nice work, you made a graph! 

In [59]:
# First make the figure
fig = go.Figure()


# Loop the columns of the dataframe and plot each as a separate trace
for col in joined_df.columns:
    fig.add_trace(
        go.Scatter(
            x=joined_df.index,
            y=joined_df[col],
            mode='lines',
            name=col
        )
    )

# Update the titles, using the html tage <sup> for a subtitle 
fig.update_layout(
    title = dict(text = 'Unemployment / 30 Year Fixed Rate Mortgage Average <br><sup>Since 2020, Monthly</sup>'),
    xaxis_title="Date",
    yaxis_title="% Change of Price / Unemployed",
    legend=dict(
        font=dict(size=15),     # smaller legend text
        orientation="h",        # horizontal (optional)
        yanchor="bottom",       # anchor position
        y=1.02,                 # place just above the plot
        xanchor="right",
        x=1
    )
)

# This is graph specific, but here we want the y-axis to be percent signs 
fig.update_yaxes(
    tickformat=".2f%",
    ticksuffix="%"
)

# Again, graph specific, we have a mutliyear series and want tick marks to be years
fig.update_xaxes(
    type='date',
    tickformat='%Y',
)

# Show our figure (Dimensions may be off on different screen sizes)
fig.show()

# This should be the same for EVERY GRAPH!
# Save it to the graph_output folder with the name matching the file, as HTML
fig.write_html(GRAPH_OUTPUT_PATH + f"/{GRAPH_NAME}.html")